In [ ]:
# Load vehicle summary data into Chroma safely

import math
import pandas as pd
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.errors import InternalError

excel_file_path = '/home/prabhu/Downloads/Summary_Report_JAN.xlsx'
persist_path = '/home/prabhu/Music/Vector_DB'
collection_name = 'vehiclesummary_jan_collection_v3'

df = pd.read_excel(excel_file_path)
model = SentenceTransformer('all-MiniLM-L6-v2')

def safe_value(v):
    if pd.isna(v):
        return ''
    if isinstance(v, float) and (math.isinf(v) or math.isnan(v)):
        return ''
    return v

texts = df.apply(
    lambda row: (
        f"Vehicle {safe_value(row['VEHICLE_ID'])} on {row['CREATED_DATE'].strftime('%Y-%m-%d')} "
        f"moved for {safe_value(row['MOVE_MINS'])} minutes, "
        f"idle for {safe_value(row['IDLE_MINS'])} minutes, "
        f"stopped for {safe_value(row['STOP_MINS'])} minutes, "
        f"and traveled {safe_value(row['DISTANCE_TRAVELED'])} km. "
        f"Reseller {safe_value(row['RESELLER_ID'])}, "
        f"Dealer {safe_value(row['DEALER_ID'])}, "
        f"Customer {safe_value(row['CUSTOMER_ID'])}, "
        f"Organization {safe_value(row['ORG_ID'])}."
    ),
    axis=1
).tolist()

metadatas = [
    {
        'VEHICLE_ID': str(safe_value(row['VEHICLE_ID'])),
        'CREATED_DATE': row['CREATED_DATE'].strftime('%Y-%m-%d %H:%M:%S'),
        'MOVE_MINS': float(safe_value(row['MOVE_MINS']) or 0),
        'IDLE_MINS': float(safe_value(row['IDLE_MINS']) or 0),
        'STOP_MINS': float(safe_value(row['STOP_MINS']) or 0),
        'RESELLER_ID': str(safe_value(row['RESELLER_ID'])),
        'DEALER_ID': str(safe_value(row['DEALER_ID'])),
        'CUSTOMER_ID': str(safe_value(row['CUSTOMER_ID'])),
        'ORG_ID': str(safe_value(row['ORG_ID'])),
        'DISTANCE_TRAVELED': float(safe_value(row['DISTANCE_TRAVELED']) or 0),
    }
    for _, row in df.iterrows()
]

# Deterministic IDs avoid duplicate-id and append-only log issues.
ids = [
    f"{str(safe_value(row['VEHICLE_ID']))}_{row['CREATED_DATE'].strftime('%Y%m%d%H%M%S')}_{i}"
    for i, (_, row) in enumerate(df.iterrows())
]

embeddings = model.encode(texts, convert_to_numpy=True).astype('float32').tolist()

def write_collection(rebuild_on_failure=False):
    client = chromadb.PersistentClient(path=persist_path)
    if rebuild_on_failure:
        try:
            client.delete_collection(collection_name)
            print(f'Rebuilt collection: {collection_name}')
        except Exception:
            pass

    collection = client.get_or_create_collection(name=collection_name)

    batch = 500
    for i in range(0, len(ids), batch):
        j = i + batch
        collection.upsert(
            ids=ids[i:j],
            documents=texts[i:j],
            metadatas=metadatas[i:j],
            embeddings=embeddings[i:j],
        )

try:
    write_collection(rebuild_on_failure=False)
except InternalError as e:
    # Typical recovery for corrupted segment/hnsw compaction state.
    if 'compaction' in str(e).lower() or 'hnsw segment writer' in str(e).lower():
        print('Compaction/index error detected. Rebuilding collection and retrying...')
        write_collection(rebuild_on_failure=True)
    else:
        raise

print('Data successfully loaded into Chroma DB!')


In [ ]:
# Load vehicle summary data into Chroma safely

import math
import pandas as pd
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.errors import InternalError

excel_file_path = '/home/prabhu/Downloads/vehicledata_collection.xlsx'
persist_path = '/home/prabhu/Music/Vector_DB'
collection_name = 'vehiclesummary_jan_collection_v3'

df = pd.read_excel(excel_file_path)
model = SentenceTransformer('all-MiniLM-L6-v2')

def safe_value(v):
    if pd.isna(v):
        return ''
    if isinstance(v, float) and (math.isinf(v) or math.isnan(v)):
        return ''
    return v

texts = df.apply(
    lambda row: (
        f"Vehicle {safe_value(row['VEHICLE_ID'])} on {row['CREATED_DATE'].strftime('%Y-%m-%d')} "
        f"moved for {safe_value(row['MOVE_MINS'])} minutes, "
        f"idle for {safe_value(row['IDLE_MINS'])} minutes, "
        f"stopped for {safe_value(row['STOP_MINS'])} minutes, "
        f"and traveled {safe_value(row['DISTANCE_TRAVELED'])} km. "
        f"Reseller {safe_value(row['RESELLER_ID'])}, "
        f"Dealer {safe_value(row['DEALER_ID'])}, "
        f"Customer {safe_value(row['CUSTOMER_ID'])}, "
        f"Organization {safe_value(row['ORG_ID'])}."
    ),
    axis=1
).tolist()

metadatas = [
    {
        'VEHICLE_ID': str(safe_value(row['VEHICLE_ID'])),
        'CREATED_DATE': row['CREATED_DATE'].strftime('%Y-%m-%d %H:%M:%S'),
        'MOVE_MINS': float(safe_value(row['MOVE_MINS']) or 0),
        'IDLE_MINS': float(safe_value(row['IDLE_MINS']) or 0),
        'STOP_MINS': float(safe_value(row['STOP_MINS']) or 0),
        'RESELLER_ID': str(safe_value(row['RESELLER_ID'])),
        'DEALER_ID': str(safe_value(row['DEALER_ID'])),
        'CUSTOMER_ID': str(safe_value(row['CUSTOMER_ID'])),
        'ORG_ID': str(safe_value(row['ORG_ID'])),
        'DISTANCE_TRAVELED': float(safe_value(row['DISTANCE_TRAVELED']) or 0),
    }
    for _, row in df.iterrows()
]

# Deterministic IDs avoid duplicate-id and append-only log issues.
ids = [
    f"{str(safe_value(row['VEHICLE_ID']))}_{row['CREATED_DATE'].strftime('%Y%m%d%H%M%S')}_{i}"
    for i, (_, row) in enumerate(df.iterrows())
]

embeddings = model.encode(texts, convert_to_numpy=True).astype('float32').tolist()

def write_collection(rebuild_on_failure=False):
    client = chromadb.PersistentClient(path=persist_path)
    if rebuild_on_failure:
        try:
            client.delete_collection(collection_name)
            print(f'Rebuilt collection: {collection_name}')
        except Exception:
            pass

    collection = client.get_or_create_collection(name=collection_name)

    batch = 500
    for i in range(0, len(ids), batch):
        j = i + batch
        collection.upsert(
            ids=ids[i:j],
            documents=texts[i:j],
            metadatas=metadatas[i:j],
            embeddings=embeddings[i:j],
        )

try:
    write_collection(rebuild_on_failure=False)
except InternalError as e:
    # Typical recovery for corrupted segment/hnsw compaction state.
    if 'compaction' in str(e).lower() or 'hnsw segment writer' in str(e).lower():
        print('Compaction/index error detected. Rebuilding collection and retrying...')
        write_collection(rebuild_on_failure=True)
    else:
        raise

print('Data successfully loaded into Chroma DB!')


In [1]:
#load vehicle_detail


import pandas as pd
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings

excel_file_path = '/home/prabhu/Downloads/vehicledata_collection.xlsx'  
df = pd.read_excel(excel_file_path)

# Step 4: Initialize the sentence-transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

print(df)

texts = df[['VEHICLE_NO', 'VEHICLE_ID','MODEL', 'RESELLER_ID','DEALER_ID','CUSTOMER_ID','ORG_ID']].apply(lambda row: ' '.join(row.values.astype(str)), axis=1).tolist()


texts = df.apply(
    lambda row: (
        f"Vehicle Number {safe_value(row['VEHICLE_NO'])}"
        f"moved for {safe_value(row['MOVE_MINS'])} minutes, "
        f"idle for {safe_value(row['IDLE_MINS'])} minutes, "
        f"stopped for {safe_value(row['STOP_MINS'])} minutes, "
        f"and traveled {safe_value(row['DISTANCE_TRAVELED'])} km. "
        f"Reseller {safe_value(row['RESELLER_ID'])}, "
        f"Dealer {safe_value(row['DEALER_ID'])}, "
        f"Customer {safe_value(row['CUSTOMER_ID'])}, "
        f"Organization {safe_value(row['ORG_ID'])}."
    ),
    axis=1
).tolist()



print(texts)

embeddings = model.encode(texts, convert_to_tensor=True)

client = chromadb.PersistentClient(path="/home/prabhu/Music/Vector_DB")
collection = client.get_or_create_collection(name="vehicle_data_collection5")

# Now you can continue with the rest of your code to create the collection
ids = df["VEHICLE_ID"].astype(str).tolist()


# Example metadata (adjust based on your DataFrame columns)
metadatas = [{'VEHICLE_ID': row['VEHICLE_ID'],'VEHICLE_NO': row['VEHICLE_NO'],  'MODEL': row['MODEL'], 'RESELLER_ID': row['RESELLER_ID'], 'DEALER_ID': row['DEALER_ID'], 'CUSTOMER_ID': row['CUSTOMER_ID'],'ORG_ID': row['ORG_ID'] } for _, row in df.iterrows()]

# Add documents, embeddings, and metadata to Chroma DB
collection.add(
        ids=ids,

    documents=texts,  # Text data
    metadatas=metadatas,  # Metadata
    embeddings=embeddings.tolist()  # Embedding vectors
)
print("Data successfully loaded into Chroma DB!")




/home/prabhu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


      VEHICLE_ID  VEHICLE_NO    MODEL  RESELLER_ID  DEALER_ID  CUSTOMER_ID  \
0          41303  CH01TB9055  Model 1       116606     116607       116614   
1          41306  CH01TB0633  Model 1       116606     116607       116614   
2          41307  CH01TB8226  Model 1       116606     116607       116614   
3          41308  CH01TB6280  Model 1       116606     116607       116614   
4          41309  CH01TB6730  Model 1       116606     116607       116614   
...          ...         ...      ...          ...        ...          ...   
1137       44289  KA01AR0380  Model 1       116606     116607       116608   
1138       44291  KA01AR1564  Model 1       116606     116607       116608   
1139       44294  KA01AR1140  Model 1       116606     116607       116608   
1140       44296  TN10RA1235  Model 1       116606     116607       116608   
1141       44297  TN10RA1236  Model 1       116606     116607       116608   

      ORG_ID  
0     116615  
1     116615  
2     116615  
3  